## Combining Isotopic age data with grain geometries (for LA-ICP-MS analyses only)
In order to combine U-Pb age and shape information for each individual zircon grain, we must determine which grain has a given isotopic age. This notebook should only be used post-ablation.

This is most easily done using a .scancsv file, created during the laser session. It is also possible to manually combine this information, if a annotated image of the grains with their analysis number is present (although this will take considerably more time).

Import libraries

In [ ]:
import cv2
import numpy as np
import pandas as pd
import re
import sez
import tkinter as tk
from tkinter import Frame

import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

from PIL import Image, ImageTk
Image.MAX_IMAGE_PIXELS = None

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from matplotlib.widgets import Button


### Loading in the scancsv file from the laser and the 2-D image that you used for segmentation

In [ ]:
scancsv_file = '/Users/omw339/Downloads/13WKEN.scancsv'  # Provide the path to your CSV file

In [ ]:
image_file = '/Users/omw339/Downloads/Untitled124/WKEN.tif' #original image

In [ ]:
laser_df = sez.load_scancsv(scancsv_file)
image = sez.load_image_post_ablation(image_file)

In [ ]:
laser_df.head(3)

In [ ]:
# Plot image
plt.imshow(image)

# Overlay scatter plot
plt.scatter(laser_df['x'], laser_df['y'], color='red', s=40)  # You can change color/size

# Optional: invert y-axis if needed (image origin is top-left)
#plt.gca().invert_yaxis()

# Show plot
plt.show()

Read in the coordinates of the grains that you segmented

In [ ]:
sez_coords_df = pd.read_csv('/Users/omw339/Desktop/summer_SEZ_outputs/WKEN/WKEN_final_coordinates_8_13_25.csv')
sez_coords_df.head()
len(sez_coords_df)

Read in the grain morphology csv file of the grains you segmented and ensure that the length of the dataframe is the same as the length of the sez_coords dataframe

In [ ]:
sez_shapes_df = pd.read_csv('/Users/omw339/Desktop/summer_SEZ_outputs/WKEN/WKEN_morphometrics.csv')
sez_shapes_df.head(4)
len(sez_shapes_df)

Combine the two dataframes

In [ ]:
sez_df = pd.concat([sez_coords_df, sez_shapes_df], axis=1)

In [ ]:

# Load CSV and parse WKT
sez_df["geometry"] = sez_df["geometry"].apply(wkt.loads)

# Convert to GeoDataFrame
sez_gdf = gpd.GeoDataFrame(sez_df, geometry="geometry")
sez_gdf.set_crs(None, inplace=True)  # It's in pixel coordinates, not geographic

# Load the image
image = Image.open('/Users/omw339/Downloads/Untitled124/WKEN.tif')
width, height = image.size

# Plot
fig, ax = plt.subplots()
ax.imshow(image, extent=[0, width, height, 0], origin='upper')
sez_gdf.plot(ax=ax, facecolor='none', edgecolor='red')

plt.show()

Inspect the newly created geodataframe to ensure the grain geometries and morphometric measurements are included

In [ ]:
sez_gdf.head(3)

In [ ]:
print(sez_gdf.total_bounds)  # [xmin, ymin, xmax, ymax]
print(sez_gdf.crs)            # Should match your image's CRS (or be known)
print(sez_gdf.is_valid.all())  # Should return True
print(sez_gdf.geometry.head()) # See actual geometry content

In [ ]:
# Load the scan CSV
laser_df["geometry_initial_coordinates"] = laser_df.apply(lambda row: Point(row["x"], row["y"]), axis=1)
laser_gdf = gpd.GeoDataFrame(laser_df, geometry="geometry_initial_coordinates")
laser_gdf.set_crs(None, inplace=True)  # Pixel coordinate system


In [ ]:
print(laser_gdf.total_bounds)  # [xmin, ymin, xmax, ymax]
print(laser_gdf.crs)            # Should match your image's CRS (or be known)
print(laser_gdf.is_valid.all())  # Should return True
print(laser_gdf.geometry.head()) # See actual geometry content

Plotting the segmented grains with the ablation spot coordinates from the scancsv. Note that it is highly unlikely that they will line up at this stage, which is why we will conduct a coordinate transformation.

In [ ]:
fig, ax = plt.subplots()
ax.imshow(image, extent=[0, width, height, 0], origin='upper')
sez_gdf.plot(ax=ax, facecolor='none', edgecolor='black')
laser_gdf.plot(ax=ax, color='red', markersize=10)
plt.show()

Add in information about the ablation size used during analysis. This is typically found in the 'Ablation Settings' section of the scancsv file

In [ ]:
laser_gdf['spot_number_µm'] = 30

converting the ablation spots from pixels to microns. 

- n_of_units = length of the scale bar. e.g. a 500 µm scalebar would be '500'
- scale_bar_length = the length of the scale bar in pixels. Measured in the morphometrics notebook

In [ ]:
n_of_units = 2000 # micrometers usually'
scale_bar_length = 1802.167  #length of scale bar in pixels
units_per_pixel = n_of_units/scale_bar_length

In [ ]:
laser_gdf["spot_radius_pix"] = (laser_gdf["spot_number_µm"] / units_per_pixel) / 2

In [ ]:
laser_gdf = laser_gdf.rename(columns={'spot_number_µm':'spot_size_µm'})

Now we will be importing our ROI set, which has a .zip extension. There are polygons drawn in ImageJ around laser ablation pits and act as "ground truth" polygons for when we do our coordinate transformation. 5-10 spots per 100 grains is usuallys sufficient.

In [ ]:
pip install roifile

In [ ]:
import re
from roifile import roiread
import geopandas as gpd
import roifile
from shapely.geometry import Polygon

# Path to the ROI .zip file
roi_zip_path = "/Users/omw339/Desktop/summer_SEZ_outputs/WKEN/RoiSet_WKEN.zip"

image = Image.open('/Users/omw339/Downloads/Untitled124/WKEN.tif')
width, height = image.size
roi_list = roifile.roiread(roi_zip_path)

# Flip settings (update as needed)
flip_vertical = False
flip_horizontal = False  # Set to True if you think X is also flipped

# Read ROI file
roi_list = roiread(roi_zip_path)

geometries = []
spot_numbers = []

for roi in roi_list:
    coords = roi.coordinates()
    if coords is None or len(coords) < 3:
        continue

    # Flip coordinates if needed
    flipped_coords = []
    for x, y in coords:
        if flip_horizontal:
            x = width - x
        if flip_vertical:
            y = height - y
        flipped_coords.append((x, y))

    # Create polygon
    poly = Polygon(flipped_coords)
    geometries.append(poly)

    # Extract spot number from ROI name (e.g., "Spot 1")
    name = roi.name
    match = re.search(r"Spot\s*(\d+)", name)
    spot_number = int(match.group(1)) if match else None
    spot_numbers.append(spot_number)

# Build GeoDataFrame
circles_gdf = gpd.GeoDataFrame({
    "spot_number": spot_numbers,
    "geometry_manual_ablation_polygon": geometries
})

# Set geometry and CRS (None, since pixel space)
circles_gdf = circles_gdf.set_geometry("geometry_manual_ablation_polygon")
circles_gdf = circles_gdf.set_crs(None)


In [ ]:
laser_gdf

In [ ]:
circles_gdf

## The Coordinate Transformation Process

### Step 1: Assigning the manual ROI polygons (the manually drawn polygons over the laser ablation pits) to grains via a spatial join

In [ ]:
# Ensure your ROI GeoDataFrame has the correct geometry set
roi_gdf = circles_gdf.set_geometry('geometry_manual_ablation_polygon')

# Ensure your grains GeoDataFrame has geometry set
grains_gdf = sez_gdf.copy()
grains_gdf = grains_gdf.rename_geometry('geometry_grain')
# Perform spatial join
roi_with_grain = gpd.sjoin(
    roi_gdf.set_geometry('geometry_manual_ablation_polygon'),
    grains_gdf,
    how='left',
    predicate='intersects'
)


# Add the grain geometry manually from seg_gdf using index_right
roi_with_grain['geometry_grain'] = roi_with_grain['index_right'].map(sez_gdf['geometry'])

In [ ]:
roi_with_grain

In [ ]:
roi_with_grain['intersection_area'] = roi_with_grain.apply(
    lambda row: row.geometry_manual_ablation_polygon.intersection(row.geometry_grain).area
    if row.geometry_grain is not None else 0,
    axis=1
)

### Step 2: Compute the Intersection Area
This insures that if an ROI overlaps with multiple grains, it will keep the one with the largest intersectional area. This step also removes ROIs that did not intersect any grain.


In [ ]:
# After sjoin, the grain geometry is usually in 'geometry_right' (check with roi_with_grain.columns)
# Rename for clarity
roi_with_grain = roi_with_grain.rename(columns={'geometry_right': 'geometry_grain'})

# Compute intersection area
# Some ROIs may not intersect any grain -> geometry_grain can be null
roi_with_grain['intersection_area'] = roi_with_grain.apply(
    lambda row: row.geometry_manual_ablation_polygon.intersection(row.geometry_grain).area
    if row.geometry_grain is not None else 0,
    axis=1
)

# If a ROI overlaps multiple grains, keep the one with the largest intersection
roi_with_grain = roi_with_grain.loc[
    roi_with_grain.groupby('spot_number')['intersection_area'].idxmax()
].reset_index(drop=True)

# Optionally, drop ROIs that did not intersect any grain
roi_with_grain = roi_with_grain[roi_with_grain['intersection_area'] > 0]

# Now roi_with_grain has one row per ROI, matched to its grain

### Step 3: Fitting the Transformation

In [ ]:
import numpy as np
from skimage.transform import estimate_transform

# Compute manual centroids
roi_with_grain['manual_centroid'] = roi_with_grain.geometry_manual_ablation_polygon.centroid

# Merge with laser stage coordinates
control_gdf = roi_with_grain.merge(
    laser_gdf[['spot_number', 'x', 'y']],  # stage coords
    on='spot_number',
    how='inner'
)

# Source: laser coords, Destination: image coords (manual centroids)
src_points = np.array(list(zip(control_gdf['x'], control_gdf['y'])))
dst_points = np.array([(pt.x, pt.y) for pt in control_gdf['manual_centroid']])

# Initial affine transform
tform_initial = estimate_transform('affine', src_points, dst_points)

# Compute residuals and exclude outliers
pred_points = tform_initial(src_points)
residuals = np.linalg.norm(pred_points - dst_points, axis=1)
control_gdf['residual'] = residuals

threshold = residuals.mean() + 2*residuals.std()
robust_control_gdf = control_gdf[control_gdf['residual'] <= threshold]

# Fit robust affine transform
src_points_robust = np.array(list(zip(robust_control_gdf['x'], robust_control_gdf['y'])))
dst_points_robust = np.array([(pt.x, pt.y) for pt in robust_control_gdf['manual_centroid']])
tform = estimate_transform('affine', src_points_robust, dst_points_robust)

### Step 4: Applying the transformation to all ablation spots

In [ ]:
# Step 4: Apply transform to all ablation spots
laser_transformed = laser_gdf.copy()

# Identify ROI pits
has_roi = laser_transformed['spot_number'].isin(circles_gdf['spot_number'])

# Non-ROI pits: apply affine transform
non_roi_idx = ~has_roi
if non_roi_idx.any():
    laser_transformed.loc[non_roi_idx, ['x_image', 'y_image']] = tform(
        laser_transformed.loc[non_roi_idx, ['x', 'y']].values
    )

# ROI pits: use manual centroids
manual_coords = robust_control_gdf.set_index('spot_number')['manual_centroid']

def get_coord(point, axis):
    if hasattr(point, 'x') and hasattr(point, 'y'):
        return point.x if axis == 'x' else point.y
    return np.nan

laser_transformed.loc[has_roi, 'x_image'] = laser_transformed.loc[has_roi, 'spot_number'].map(
    lambda s: get_coord(manual_coords.get(s, np.nan), 'x')
)
laser_transformed.loc[has_roi, 'y_image'] = laser_transformed.loc[has_roi, 'spot_number'].map(
    lambda s: get_coord(manual_coords.get(s, np.nan), 'y')
)

# Drop pits without valid coordinates (unsegmented grains / missing manual centroids)
laser_transformed = laser_transformed.dropna(subset=['x_image', 'y_image']).copy()

### Step 5: Creating a spots with grains geodataframe that matches all of the laser ablation spots to the appropriate grain/

In [ ]:
# Convert to GeoDataFrame and assign pits to grains
spots_gdf = gpd.GeoDataFrame(
    laser_transformed,
    geometry=gpd.points_from_xy(laser_transformed.x_image, laser_transformed.y_image),
    crs=sez_gdf.crs
)

# Spatial join: assign pits to grains
spots_with_grains = gpd.sjoin(
    spots_gdf,
    sez_gdf,
    how='inner',  # discard pits not overlapping any grain
    predicate='intersects'
)

spots_with_grains = spots_with_grains.drop(columns=['index_right'])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot grain boundaries
sez_gdf.boundary.plot(ax=ax, color='gray', linewidth=0.8, label='Grains')

# Plot manual ROI pit polygons (skip missing geometries)
roi_valid = roi_with_grain[roi_with_grain.geometry_manual_ablation_polygon.notnull()]
roi_valid.set_geometry('geometry_manual_ablation_polygon').plot(
    ax=ax, color='red', alpha=0.5, edgecolor='darkred', label='Manual ROI pits'
)

# Plot non-ROI pits (transformed points)
non_roi_pits = spots_with_grains[~spots_with_grains['spot_number'].isin(circles_gdf['spot_number'])]
# Ensure geometry exists
non_roi_pits = non_roi_pits[non_roi_pits.geometry.notnull()]
non_roi_pits.plot(ax=ax, color='blue', markersize=30, label='Non-ROI transformed pits')

# Plot centroids of ROI polygons
roi_valid = roi_valid.copy()
roi_valid['centroid'] = roi_valid.geometry_manual_ablation_polygon.centroid
roi_centroids = roi_valid.set_geometry('centroid')
roi_centroids.plot(ax=ax, color='darkred', markersize=20, marker='x', label='ROI centroids')

ax.legend()
ax.set_title('Grains, Manual ROI Pits, and Transformed Non-ROI Pits')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.show()

In [ ]:
spots_with_grains

## Adding the Isotopic Data
Now Combining the U-Pb Data with the Grains

In [ ]:
isotopic_age_data = pd.read_csv('/Users/omw339/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/OWBP25012_Age_Data_Publication.csv')

In [ ]:
isotopic_age_data

In [ ]:
isotopic_age_data = isotopic_age_data.dropna(how='all')

In [ ]:
isotopic_age_data['spot_number'] = isotopic_age_data['Grain_ID'].str.split('_').str[-1].astype('Int64')
isotopic_age_data

In [ ]:
merged_df = pd.merge(spots_with_grains, isotopic_age_data, on='spot_number', how='inner')
merged_df

In [ ]:
merged_df.to_csv('OWBP25012_shapes_with_dates.csv')